In [1]:
import numpy as np
import matplotlib.pyplot as plt
import corner
import warnings
from datetime import datetime
from VegasAfterglow import Fitter, ParamDef, Scale
from VegasAfterglow.units import keV, mJy, hr, sec, Hz, Jy, _c_A
from astropy.cosmology import FlatLambdaCDM
from grb.io import read_data, filter_data
from grb.utils import mJy_to_erg_cm2_s
from grb.const import REDSHIFT, RA, DEC, TRIGGER_TIME, AV, FILTER_INFO

warnings.filterwarnings('ignore', category=FutureWarning) 
#%%
# ===========================================================================
## Environment setup ##
# ===========================================================================

# Constants
Z = REDSHIFT
RA = RA
DEC = DEC
T0 = TRIGGER_TIME
AV = AV
WAVELENGTH_MAP = FILTER_INFO
C_LIGHT = _c_A / 10  # Speed of light in nm per second

# Current Timestamp
_RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

# XRT energy range
XRT_NU_MIN = 0.3 * keV  
XRT_NU_MAX = 10 * keV  
XRT_NU = 10 * keV  

# Calculate luminosity distance
cosmo = FlatLambdaCDM(H0=67.66, Om0=0.27) # Use same value from VegasAfterglow
LUMI_DIST = cosmo.luminosity_distance(Z).to('cm').value

# Generate model light curves and spectrum
LC_T_RANGE    = np.logspace(2, 6, 200)                           # 10 s → 10^6 s
SPEC_F_RANGE  = np.logspace(14, 15, 200)                         # 10^14 Hz → 10^15 Hz

# Band style for plotting
BAND_STYLE = {
    XRT_NU:                                                      ("black",       "X-ray"),
    C_LIGHT / WAVELENGTH_MAP["g"]["central_wavelength_nm"]:      ("teal",        "g"),
    C_LIGHT / WAVELENGTH_MAP["r"]["central_wavelength_nm"]:      ("crimson",     "r"),
    C_LIGHT / WAVELENGTH_MAP["i"]["central_wavelength_nm"]:      ("darkviolet",  "i"),
    C_LIGHT / WAVELENGTH_MAP["Ic"]["central_wavelength_nm"]:     ("indigo",      "Ic"),
    C_LIGHT / WAVELENGTH_MAP["Rc"]["central_wavelength_nm"]:     ("orangered",   "Rc"),
    C_LIGHT / WAVELENGTH_MAP["V"]["central_wavelength_nm"]:      ("forestgreen", "V"),
    C_LIGHT / WAVELENGTH_MAP["J"]["central_wavelength_nm"]:      ("saddlebrown", "J"),
    C_LIGHT / WAVELENGTH_MAP["clear"]["central_wavelength_nm"]:  ("darkgrey",    "clear"),
}

# Extract frequencies[Hz] from BAND_STYLE
LC_BANDS = np.array(list(BAND_STYLE.keys()))

#%%
# ===========================================================================
## Model configuration ##
# ===========================================================================

# File name
FILE_NAME = "twojet_dynesty"

# Fitter with model configuration
fitter = Fitter(
    # Source properties
    z=Z,                   # Redshift
    lumi_dist=LUMI_DIST,       # Luminosity distance [cm]

    # Model selection (see sections below for all options)
    jet="two_component",           # Jet structure type
    medium="wind",            # Ambient medium type

    # Physics options
    rvs_shock=False,           # Include reverse shock
    fwd_ssc=False,             # Forward shock inverse Compton
    rvs_ssc=False,            # Reverse shock inverse Compton
    kn=False,                  # Klein-Nishina corrections
    magnetar=False,            # Magnetar energy injection

    # Numerical parameters
    rtol=1e-5,                # Numerical tolerance
    resolution=(0.15, 0.5, 10),  # Grid resolution (phi, theta, t)
)

# Basic parameter set
params = [
    # Jet parameters
    ParamDef("E_iso",   1e50,  1e55,  Scale.log),     # Isotropic energy in erg
    ParamDef("Gamma0",   100,   500,  Scale.log),     # Lorentz factor
    ParamDef("theta_c", 0.01,   0.5,  Scale.linear),  # Opening angle in radians
    ParamDef("theta_v",    0,   0.3,  Scale.fixed),   # Viewing angle (on-axis) in radians
    
    # Medium parameters
    # ParamDef("n_ism",   1e-3,   100,  Scale.log),     # Number density in cm^-3
    
    # Wind medium parameter (replaces n_ism)
    ParamDef("A_star",  1e-3,   10,  Scale.log),     # Wind parameter
    
    # Microphysics parameters
    ParamDef("p",        2.1,   3.0,  Scale.linear),  # Electron spectral index
    ParamDef("eps_e",   1e-3,   0.5,  Scale.log),     # Electron energy fraction
    ParamDef("eps_B",   1e-5,   0.1,  Scale.log),     # Magnetic energy fraction
    ParamDef("xi_e",     0.01,   1.0,  Scale.linear),  # Fraction of accelerated electrons
    
    # Wide component
    ParamDef("E_iso_w", 1e49,  1e52,  Scale.log),     # Wide energy in erg
    ParamDef("Gamma0_w",  10,   100,  Scale.log),     # Wide Lorentz factor
    ParamDef("theta_w",  0.1,   0.5,  Scale.linear),  # Wide angle in radians
    
    # Jet duration (important for reverse shock)
    # ParamDef("tau",        1,   1e6,  Scale.log),     # Jet duration in seconds
    
    # # Reverse shock microphysics (can be different)
    # ParamDef("p_r",      2.1,   2.8,  Scale.linear),
    # ParamDef("eps_e_r", 1e-3,   0.5,  Scale.log),
    # ParamDef("eps_B_r", 1e-5,   0.1,  Scale.log),
    # ParamDef("xi_e_r",   0.1,   1.0,  Scale.linear),
    
    # Magnetar parameters (if magnetar=True)
    # ParamDef("B",        1e15,  1e16, Scale.log),     # Magnetic field strength in G
    # ParamDef("P",        0.5,   1.0,  Scale.linear),  # Period in seconds
    # ParamDef("Mdot",     1e14,  1e16, Scale.log),     # Mass accretion rate in g/s
]

#%%
# ===========================================================================
## Data preparation ##
# ===========================================================================
print("=" * 50)
print("Preparing data...")
print("=" * 50)
##################### XRT lightcurve #####################
xrt_data = read_data("xrt")
# xrt_data = read_data("xrt_unabsorb")

filtered_xrt_data = filter_data(xrt_data, exclude_time_range = (3e3,  1e4)) # Exclude flare

xrt_err = np.array([max(np.abs(high), np.abs(low)) for high, low in zip(filtered_xrt_data["Flux_high"], filtered_xrt_data["Flux_low"])])
# # Debug check for XRT
# xrt_zero_mask = (xrt_err == 0.0) | np.isnan(xrt_err)
# if np.any(xrt_zero_mask):
#     print(f"WARNING: Found {np.sum(xrt_zero_mask)} XRT points with 0.0 error!")
#     print(filtered_xrt_data.loc[xrt_zero_mask, ["Time", "Flux", "Flux_high", "Flux_low"]])

fitter.add_flux(band=(XRT_NU_MIN, XRT_NU_MAX),
                t=filtered_xrt_data["Time"] * sec,
                flux=filtered_xrt_data["Flux"], # erg/cm^2/s
                err=xrt_err, # erg/cm^2/s
                weights=None)  # All quantities in CGS units

# fitter.add_flux_density(nu=XRT_NU,
#                 t=filtered_xrt_data["Time"] * sec,
#                 f_nu=filtered_xrt_data["Flux"] * Jy, err=xrt_err * Jy,
#                 weights=None)  # All quantities in CGS units
print("XRT data added")

##################### 7DT SED #####################
sdt_data = read_data("sdt_pivot", correct_galactic_extinction=True, correct_host_extinction=True, 
                     host_av=AV, host_z=Z, add_converted_flux=True)
sdt_time = sdt_data["date_obs"].iloc[0]
sdt_from_t0 = datetime.strptime(sdt_time, '%Y-%m-%dT%H:%M:%S.%f') - T0
SDT_SECONDS = sdt_from_t0.total_seconds()
fitter.add_spectrum(t=SDT_SECONDS * sec, 
                    nu=sdt_data["frequency_Hz"] * Hz,
                    f_nu=sdt_data["flux_mJy"]* mJy, # erg/cm^2/s/Hz
                    err=sdt_data["flux_error_mJy"]* mJy)  # All quantities in CGS units
print("7DT SED added")

##################### GCN Circular #####################
circular_data = read_data("circular_wavelength", correct_galactic_extinction=True, correct_host_extinction=True, 
                     host_av=AV, host_z=Z, add_converted_flux=True)

# Filter configurations
filter_configs = [
    {"filter": "g", "facility": "7DT"},
    {"filter": "r'", "facility": "OHP/T193"},
    {"filter": "i", "facility": "NUTTelA-TAO"},
    {"filter": "Ic", "facility": "Leavitt"},
    {"filter": "Rc", "facility": "Leavitt"},
    {"filter": "V", "facility": "OsservatorioAstronomicoNastroVerde"},
    {"filter": "J", "facility": "SYSU"},
    {"filter": "clear", "facility": "Calapai"},
]

select_circular_data = []
for config in filter_configs:
    df = filter_data(
        circular_data, 
        filter_name=config["filter"], 
        facility_name=config["facility"], 
        remove_upper_limits=True, 
        exclude_time_range=(50, float('inf'))
    )
    
    if df.empty:
        print(f"No data for {config['filter']} from {config['facility']}, skipping...")
        continue
    
    select_circular_data.append(df)
    
    fitter.add_flux_density(
        nu=df["frequency_Hz"].to_numpy(), # Passing the whole array
        t=df["Time"].to_numpy() * hr,
        f_nu=df["flux_mJy"].to_numpy() * mJy,
        err=df["flux_error_mJy"].to_numpy() * mJy,
        weights=None
    )
    
    print(f"Added {len(df)} points for {config['filter']} filter ({config['facility']})")

# %%
# ===========================================================================
## Fit the model ##
# ===========================================================================
print("=" * 50)
print("Starting fitting...")
print("=" * 50)
# result = fitter.fit(
#     params,
#     resolution=(0.1, 0.25, 10),      # Grid resolution (phi, theta, t)
#     sampler="emcee",             # Nested sampling algorithm
#     nsteps=10000,                    # Number of steps
#     nburn=2000,                     # Number of burn-in steps
#     npool=20,                       # Number of parallel threads
#     top_k=10,                      # Number of best-fit parameters to return
#     resume=False,                    # Resume previous run
#     label=FILE_NAME
# )
result = fitter.fit(
    params,
    resolution=(0.1, 0.25, 10),      # Grid resolution (phi, theta, t)
    sampler="dynesty",             # Nested sampling algorithm
    nlive=1000,                    # Number of live points
    walks=100,                     # Number of random walks per live point
    dlogz=0.5,                     # Stopping criterion (evidence tolerance)
    npool=20,                       # Number of parallel threads
    top_k=10,                      # Number of best-fit parameters to return
    label=FILE_NAME,
    resume=True
)
print("Fitting completed!")

# Print best-fit parameters
print("Top-k parameters:")
header = f"{'Rank':>4s}  {'chi^2':>10s}  " + "  ".join(f"{name:>10s}" for name in result.labels)
print(header)
print("-" * len(header))
for i in range(result.top_k_params.shape[0]):
    chi2 = -2 * result.top_k_log_probs[i]
    vals = "  ".join(f"{val:10.4f}" for val in result.top_k_params[i])
    print(f"{i+1:4d}  {chi2:10.2f}  {vals}")

lc_model       = np.asarray(fitter.flux(result.top_k_params[0], 
                                                     LC_T_RANGE, # seconds
                                                     band=(XRT_NU_MIN, XRT_NU_MAX)))  # Hz
spec_model     = np.asarray(fitter.flux_density_grid(result.top_k_params[0], 
                                                     [SDT_SECONDS], # seconds
                                                     SPEC_F_RANGE))  # Hz
# %%
# ===========================================================================
## Plotting ##
# ===========================================================================
print("=" * 50)
print("Generating plots...")
print("=" * 50)
flat_chain = result.samples.reshape(-1, result.samples.shape[-1])

# Corner plot for parameter correlations
fig = corner.corner(
    flat_chain,
    labels=result.latex_labels,
    quantiles=[0.16, 0.5, 0.84],
    show_titles=True,
    title_kwargs={"fontsize": 12},
    label_kwargs={"fontsize": 14},
    truths=np.median(flat_chain, axis=0),  # Show median values
    truth_color="red",
    bins=30,
    smooth=1,
    fill_contours=True,
    levels=[0.16, 0.5, 0.68],  # 1σ and 2σ contours
    color="k"
)
plt.savefig(f"corner_plot{_RUN_TS}_{FILE_NAME}.png", dpi=300, bbox_inches='tight')
print(f"Corner plot saved as corner_plot{_RUN_TS}_{FILE_NAME}.png")

#%%
# Light curve comparison
def draw_bestfit(lc_model=None, spec_model=None, save=True):
    """
    lc_model     : shape [n_bands, n_t] from flux_density_grid
    spec_model   : shape [n_nu, 1] from flux_density_grid at t_spec_sec
    """
    fig = plt.figure(figsize=(4.5, 7.5))
    ax1 = fig.add_subplot(211)
    ax2 = fig.add_subplot(212)

    # ── Light-curve model
    if lc_model is not None:
        for i, nu in enumerate(LC_BANDS):
            color, label = BAND_STYLE.get(nu, ("grey", f"{nu:.2e} Hz"))
            ax1.plot(LC_T_RANGE/3600, lc_model[i], "-", color=color, lw=1.5, label=f"model {label}")

    # Observed XRT (approximate flux density for display)
    ax1.errorbar(filtered_xrt_data["Time"]/3600, # hours
                 filtered_xrt_data["Flux"], xrt_err, # erg/cm^2/sec
                 fmt=".", color="blue", markersize=4,
                 markeredgecolor="k", markeredgewidth=0.4, label="XRT")

    # Observed optical (GCN circulars) — only bands that have a model curve
    for data in select_circular_data:
        if data.empty:
            print(f"Skipping empty data - {data}")
            continue
        wave = data["wavelength"].iloc[0]
        freq = data["frequency_Hz"].iloc[0]
        closest_freq = min(BAND_STYLE.keys(), key=lambda k: abs(k - freq))
        
        if abs(closest_freq - freq) / freq > 0.01:
            print(f"Skipping {data['Filter'].iloc[0]}/{wave} A filter - not in BAND_STYLE")
            continue

        color, label = BAND_STYLE[closest_freq]
        ax1.errorbar(data["Time"].to_numpy(), # hours
                     mJy_to_erg_cm2_s(data["flux_mJy"].to_numpy(), closest_freq), mJy_to_erg_cm2_s(data["flux_error_mJy"].to_numpy(), closest_freq), # erg/cm^2/sec
                     fmt=".", color=color, markersize=4,
                     markeredgecolor="k", markeredgewidth=0.4, label=f"{label}")
        
    # ax1.set_xlim(LC_T_RANGE.min()/3600, LC_T_RANGE.max()/3600)
    ax1.set_xscale("log")
    ax1.set_yscale("log")
    
    ax1.set_xlabel("t [h]")
    ax1.set_ylabel(r"$F$ [erg/cm$^2$/s]")
    ax1.set_title("Light Curve Fitting")
    ax1.legend(fontsize=7)

    # ── Spectrum model
    if spec_model is not None:
        ax2.plot(SPEC_F_RANGE, spec_model[:, 0], "-", color="k", lw=1.5,
                label=f"model at t={SDT_SECONDS/3600:.2f} h")
    
    # 7DT SED
    freq = sdt_data["frequency_Hz"]
    # ax2.errorbar(freq, # Hz
    #              mJy_to_erg_cm2_s(sdt_data["flux_mJy"], freq), mJy_to_erg_cm2_s(sdt_data["flux_error_mJy"], freq), # erg/cm^2/s
    #              fmt="o", color="purple", markersize=5,
    #              markeredgecolor="k", markeredgewidth=0.4, label="7DT SED")
    ax2.errorbar(freq, # Hz
                sdt_data["flux_mJy"] * mJy, sdt_data["flux_error_mJy"] * mJy, # erg/cm^2/s
                fmt="o", color="purple", markersize=5,
                markeredgecolor="k", markeredgewidth=0.4, label="7DT SED")
    
    # ax2.set_xlim(SPEC_F_RANGE.min(), SPEC_F_RANGE.max())
    ax2.set_xscale("log")
    ax2.set_yscale("log")

    ax2.set_xlabel(r"$\nu$ [Hz]")   
    # ax2.set_ylabel(r"$F$ [erg/cm$^2$/s]")
    ax2.set_ylabel(r"$F_\nu$ [erg/cm$^2$/s/Hz]")
    ax2.set_title(f"7DT SED at t={SDT_SECONDS/3600:.2f} h")
    ax2.legend(fontsize=7)

    plt.tight_layout()
    if save:
        plt.savefig(f"bestfit_{_RUN_TS}_{FILE_NAME}.png", dpi=300, bbox_inches="tight")
        print(f"Best fit plot saved as bestfit_{_RUN_TS}_{FILE_NAME}.png")
    else:
        plt.show()

draw_bestfit(lc_model=lc_model, spec_model=spec_model, save=False)




ZeroDivisionError: float division by zero